# Tenacious-Bench ORPO Judge Training

**Trains Qwen2.5-1.5B-Instruct** with LoRA using ORPO on Tenacious-specific rubric preference pairs.

Runtime: T4 GPU (Colab free tier)  
Expected training time: ~45-90 minutes for 3 epochs

## Setup
1. Set HF_TOKEN and OPENROUTER_API_KEY in Colab Secrets (key icon in left sidebar)
2. Run all cells in order


In [ ]:
# Step 1: Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500])

In [ ]:
# Step 2: Install Unsloth and dependencies (pinned versions)
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q trl==0.12.2 peft==0.14.0 transformers==4.47.1 datasets==3.2.0
!pip install -q accelerate==1.2.1 bitsandbytes==0.45.0
print('Installation complete')

In [ ]:
# Step 3: Clone the repo
import os
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY

!git clone https://huggingface.co/datasets/rafiakedir/tenacious-bench-v0.1 /content/tenacious-bench-data
print('Repo cloned')

In [ ]:
# Step 4: Load preference pairs
import json
from pathlib import Path

pairs_path = Path('/content/tenacious-bench-data/training_data/preference_pairs.jsonl')
pairs = []
with open(pairs_path) as f:
    for line in f:
        p = json.loads(line)
        pairs.append({'prompt': p['prompt'], 'chosen': p['chosen'], 'rejected': p['rejected']})

print(f'Loaded {len(pairs)} preference pairs')
print(f'Sample pair task context (first 200 chars of prompt):')
print(pairs[0]['prompt'][:200])

In [ ]:
# Step 5: Load Unsloth model with 4-bit quantization
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024
DTYPE = None  # auto-detect
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-1.5B-Instruct',
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print('Model loaded')

In [ ]:
# Step 6: Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'v_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA applied')

In [ ]:
# Step 7: Set up ORPO trainer
import random
import numpy as np
from datasets import Dataset
from trl import ORPOConfig, ORPOTrainer

# Fixed seed
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Detect precision
cap = torch.cuda.get_device_capability()
use_fp16 = cap[0] < 8  # T4 uses fp16
use_bf16 = cap[0] >= 8  # A100/4090 use bf16
print(f'GPU compute capability: {cap}, fp16={use_fp16}, bf16={use_bf16}')

dataset = Dataset.from_list(pairs)

training_args = ORPOConfig(
    output_dir='/content/tenacious-adapter',
    learning_rate=8e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    beta=0.1,
    max_length=1024,
    max_prompt_length=512,
    logging_steps=10,
    save_steps=50,
    seed=42,
    fp16=use_fp16,
    bf16=use_bf16,
    report_to='none',
    remove_unused_columns=False,
)

trainer = ORPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)
print('Trainer initialized')

In [ ]:
# Step 8: Train
print('Starting ORPO training...')
train_result = trainer.train()
print(f'Training complete!')
print(f'Metrics: {train_result.metrics}')

In [ ]:
# Step 9: Plot loss curve
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps = [x['step'] for x in log_history if 'loss' in x]
losses = [x['loss'] for x in log_history if 'loss' in x]

if steps:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, 'b-', linewidth=2, label='Training Loss')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('ORPO Training Loss — Tenacious Judge')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('/content/loss_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Initial loss: {losses[0]:.4f}')
    print(f'Final loss:   {losses[-1]:.4f}')
    print(f'Loss decrease: {losses[0] - losses[-1]:.4f} ({(1-losses[-1]/losses[0])*100:.1f}%)')
else:
    print('No loss history available')

In [ ]:
# Step 10: Save adapter locally and push to HuggingFace
ADAPTER_DIR = '/content/tenacious-adapter'

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved to {ADAPTER_DIR}')

# Push to HuggingFace
HUB_MODEL_ID = 'rafiakedir/tenacious-bench-adapter'
print(f'Pushing to {HUB_MODEL_ID}...')
model.push_to_hub(HUB_MODEL_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HUB_MODEL_ID, token=HF_TOKEN)
print(f'Adapter pushed to https://huggingface.co/{HUB_MODEL_ID}')

In [ ]:
# Step 11: Verify adapter on HuggingFace
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
files = api.list_repo_files(HUB_MODEL_ID)
print(f'Files in {HUB_MODEL_ID}:')
for f in files:
    print(f'  {f}')

In [ ]:
# Step 12: Quick smoke test — run judge on one sample
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

JUDGE_SYSTEM = (
    'You are evaluating outbound sales emails for Tenacious Consulting. '
    'Score the following output on signal-grounding fidelity, bench commitment honesty, '
    'ICP segment appropriateness, and Tenacious tone adherence. '
    'Return JSON: {\"signal_grounding\": 0-1, \"bench_honesty\": 0-1, \"icp_segment\": 0-1, \"tone\": 0-1, \"overall\": 0-1}'
)

test_email = '''Subject: TalentBridge's ML hiring + 30-min call\n\nHi Casey,\nTalentBridge recently closed a Series A and currently has 8 open ML roles.\nWe staff ML squads, typically 4 engineers in under 3 weeks.\nWant to set up a 30-minute scoping conversation?\n\nBest,\nYabi'''

prompt_text = (
    f'<|im_start|>system\n{JUDGE_SYSTEM}<|im_end|>\n'
    f'<|im_start|>user\n{test_email}<|im_end|>\n'
    f'<|im_start|>assistant\n'
)

inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=100, temperature=0.0, do_sample=False)
generated = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print('Judge output:')
print(generated)